# 02 - RFM Analysis

Covers sections 8 to 10 of the project: customer-level features, RFM scoring, and RFM visualization.
Input: `data/clean_transactions.parquet` (output of `01_cleaning_eda.ipynb`).

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

df = pd.read_parquet("../data/clean_transactions.parquet")

df_customers = df[df["CustomerID"].notna()].copy()

print("Rows:", len(df))
print("Registered-customer rows:", len(df_customers))
print("Customers:", df_customers["CustomerID"].nunique())
print("Date range:", df["InvoiceDate"].min(), "->", df["InvoiceDate"].max())

Rows: 531235
Registered-customer rows: 399655
Customers: 4361
Date range: 2010-12-01 08:26:00 -> 2011-12-09 12:50:00


## 8. Customer-Level Features

One row per registered customer. Recency and Frequency use sales invoices only (cancellations are not purchases). Monetary is net revenue (sales minus cancellations).

In [2]:
sales = df_customers[~df_customers["IsCancellation"]]
canc = df_customers[df_customers["IsCancellation"]]

snapshot = df["InvoiceDate"].max().normalize() + pd.Timedelta(days=1)
print("Snapshot date:", snapshot.date())

per_inv = sales.groupby(["CustomerID", "InvoiceNo"]).agg(
    revenue=("TotalLineRevenue", "sum"),
    units=("Quantity", "sum"),
    lines=("StockCode", "size"),
    date=("InvoiceDate", "min"),
).reset_index()

feat = per_inv.groupby("CustomerID").agg(
    frequency=("InvoiceNo", "nunique"),
    gross_revenue=("revenue", "sum"),
    avg_order_value=("revenue", "mean"),
    avg_units_per_order=("units", "mean"),
    avg_lines_per_order=("lines", "mean"),
    first_purchase=("date", "min"),
    last_purchase=("date", "max"),
)

feat["recency"] = (snapshot - feat["last_purchase"].dt.normalize()).dt.days
feat["tenure_days"] = (snapshot - feat["first_purchase"].dt.normalize()).dt.days
feat["cancelled_value"] = -canc.groupby("CustomerID")["TotalLineRevenue"].sum()
feat["cancelled_value"] = feat["cancelled_value"].fillna(0)
feat["monetary"] = feat["gross_revenue"] - feat["cancelled_value"]
feat["monetary"] = feat["monetary"].round(4)
feat["is_uk"] = (df_customers.groupby("CustomerID")["Country"]
                 .agg(lambda s: s.mode().iloc[0]) == "United Kingdom").astype(int)

feat = feat.reset_index()

print("Customers with at least one sale:", len(feat))
print("Monetary <= 0:", (feat["monetary"] <= 0).sum())
print(feat[["recency", "frequency", "monetary"]].describe(percentiles=[.5, .9]).round(1))
feat.head()

Snapshot date: 2011-12-10
Customers with at least one sale: 4333
Monetary <= 0: 13
       recency  frequency  monetary
count   4333.0     4333.0    4333.0
mean      93.2        4.2    1908.5
std      100.1        7.6    8292.0
min        1.0        1.0    -141.5
50%       51.0        2.0     646.9
90%      264.0        9.0    3449.7
max      374.0      206.0  278778.0


,CustomerID,frequency,gross_revenue,avg_order_value,avg_units_per_order,avg_lines_per_order,first_purchase,last_purchase,recency,tenure_days,cancelled_value,monetary,is_uk
0,12347,7,4310.00,615.714286,351.142857,26.00,2010-12-07 14:57:00,2011-12-07 15:52:00,3,368,0.00,4310.00,0
1,12348,4,1437.24,359.310000,583.000000,6.75,2010-12-16 19:09:00,2011-09-25 13:13:00,76,359,0.00,1437.24,0
2,12349,1,1457.55,1457.550000,630.000000,72.00,2011-11-21 09:51:00,2011-11-21 09:51:00,19,19,0.00,1457.55,0
3,12350,1,294.40,294.400000,196.000000,16.00,2011-02-02 16:01:00,2011-02-02 16:01:00,311,311,0.00,294.40,0
4,12352,7,1385.74,197.962857,75.142857,11.00,2011-02-16 12:33:00,2011-11-03 14:37:00,37,297,120.33,1265.41,0


In [3]:
feat[feat["monetary"] <= 0][
    ["CustomerID", "frequency", "gross_revenue", "cancelled_value", "monetary", "recency"]
]

,CustomerID,frequency,gross_revenue,cancelled_value,monetary,recency
88,12454,1,3528.34,3528.34,0.00,57
167,12558,1,269.96,269.96,0.00,8
209,12607,1,1579.51,1579.51,0.00,61
766,13364,1,134.96,134.96,0.00,72
982,13672,2,186.39,186.39,0.00,302
1049,13762,1,306.72,306.72,0.00,219
1628,14557,1,788.38,788.38,0.00,86
1792,14792,1,6.20,6.20,0.00,64
3075,16546,2,787.15,883.08,-95.93,291
3319,16878,1,13.30,13.30,0.00,85


In [4]:
feat_nonpos = feat[feat["monetary"] <= 0].copy()
feat = feat[feat["monetary"] > 0].copy().reset_index(drop=True)

print("Isolated (monetary <= 0):", len(feat_nonpos))
print("Customers for RFM:", len(feat))
print("Net revenue of isolated customers:", round(feat_nonpos["monetary"].sum(), 2))

Isolated (monetary <= 0): 13
Customers for RFM: 4320
Net revenue of isolated customers: -237.41


**Decision 8:** 13 customers with net `monetary` <= 0 were isolated into `feat_nonpos` and excluded from RFM and clustering. Eleven are exact zero or negative (nine returned their entire purchase; two are negative at -95.93 and -141.48, possibly from purchases made before the dataset starts, an assumption). Two more (customers 12454 and 13762) showed a net of ~1e-13 GBP caused by floating-point subtraction of identical amounts, so `monetary` is rounded to 2 decimals before the filter. Together the isolated customers net -237.41 GBP (-0.003% of total monetary), so excluding them does not change the analysis. Log transforms and quantile scoring require positive values. Customers with only cancellations (28) were already excluded because they have no purchase to compute Recency from.

## 9. RFM Analysis

Each customer is scored 1 to 5 on Recency, Frequency, and Monetary using `pd.qcut` (5 equal-sized groups). Recency is reversed: fewer days since the last purchase gives a higher score (5 = most recent). Scores are combined into a string such as `'555'`.

In [5]:
rfm = feat[["CustomerID", "recency", "frequency", "monetary"]].copy()

# Recency: lower days = higher score (labels reversed)
rfm["R"] = pd.qcut(rfm["recency"].rank(method="first"), 5, labels=[5, 4, 3, 2, 1]).astype(int)

# Frequency and Monetary: higher value = higher score
rfm["F"] = pd.qcut(rfm["frequency"].rank(method="first"), 5, labels=[1, 2, 3, 4, 5]).astype(int)
rfm["M"] = pd.qcut(rfm["monetary"].rank(method="first"), 5, labels=[1, 2, 3, 4, 5]).astype(int)

rfm["RFM_Score"] = rfm["R"].astype(str) + rfm["F"].astype(str) + rfm["M"].astype(str)
rfm["RFM_Sum"] = rfm["R"] + rfm["F"] + rfm["M"]

print("Customers scored:", len(rfm))
print("Distinct RFM strings:", rfm["RFM_Score"].nunique())
print("\nCustomers per score (should be ~equal):")
print(rfm[["R", "F", "M"]].apply(lambda s: s.value_counts().sort_index()))

print("\nRaw value range per score:")
for col, raw in [("R", "recency"), ("F", "frequency"), ("M", "monetary")]:
    print(f"\n{col}:")
    print(rfm.groupby(col)[raw].agg(["min", "median", "max"]).round(1))

print("\nTop 10 most common RFM strings:")
print(rfm["RFM_Score"].value_counts().head(10))

rfm.head()

Customers scored: 4320
Distinct RFM strings: 119

Customers per score (should be ~equal):
     R    F    M
1  864  864  864
2  864  864  864
3  864  864  864
4  864  864  864
5  864  864  864

Raw value range per score:

R:
   min  median  max
R                  
1  180   264.0  374
2   72   109.0  180
3   33    51.0   72
4   15    23.0   33
5    1     6.0   15

F:
   min  median  max
F                  
1    1     1.0    1
2    1     1.0    2
3    2     2.0    3
4    3     4.0    6
5    6     9.0  206

M:
      min  median       max
M                          
1     2.9   152.8     242.4
2   242.4   340.5     471.4
3   471.6   650.4     908.9
4   910.0  1311.4    1998.5
5  2000.3  3455.9  278778.0

Top 10 most common RFM strings:
RFM_Score
555    346
111    187
455    174
121    153
112    120
444    114
122     93
233     88
344     87
544     85
Name: count, dtype: int64


,CustomerID,recency,frequency,monetary,R,F,M,RFM_Score,RFM_Sum
0,12347,3,7,4310.00,5,5,5,555,15
1,12348,76,4,1437.24,2,4,4,244,10
2,12349,19,1,1457.55,4,1,4,414,9
3,12350,311,1,294.40,1,1,2,112,4
4,12352,37,7,1265.41,3,5,4,354,12


**Decision 9:** Customers (4,320) are scored 1 to 5 with `pd.qcut` on `rank(method="first")` of each metric, giving 864 customers per score. Ranking before `qcut` is needed because Frequency has many ties (34.7% of customers have a single order), which would otherwise create duplicate bin edges.

**Score meaning:** R5 = 1-15 days since last purchase, R1 = 180-374 days; F5 = 6+ invoices, F1 = 1 invoice; M5 >= ~2,000 GBP, M1 < ~242 GBP.

**Known limitations of this scoring:**
- Customers with identical Frequency can fall in adjacent scores (Frequency 1 appears in F=1 and F=2), so F scores 1 to 3 are approximate.
- M=5 is very wide (about 2,000 to 278,778 GBP), so extreme customers are not distinguished by score alone.
- R, F, and M are correlated, so combinations cluster in the corners (119 of 125 strings occur, but a few dominate).
- Bin edges come from the data distribution, not from business-defined thresholds.

Final segments will come from clustering (Section 12), with RFM scores used for comparison.

### Insights 9 (RFM scoring)

40. **Scores are balanced by construction.** Each score from 1 to 5 holds exactly 864 customers (4,320 scored).
41. **Score thresholds.** Recency: 5 = 1-15 days, 4 = 15-33, 3 = 33-72, 2 = 72-180, 1 = 180-374. Monetary: 1 < ~242 GBP, 2 = ~242-471, 3 = ~471-909, 4 = ~909-1,998, 5 >= ~1,998 GBP.
42. **The most common profile is the top one.** `555` has 346 customers (8.0%), the largest group, followed by `455` with 174. The scores are correlated, so customers concentrate in the corners.
43. **Frequency scores are coarse at the low end.** A single invoice appears in both F=1 and F=2 because of the tie-breaking rank.

Findings are descriptive; RFM scores are a scoring convention, not behavioral segments.

## 10. RFM Visualization & Interpretation

Score distributions, the Recency x Frequency relationship, and the revenue share of each score group.

In [6]:
# Revenue share by score
for col in ["R", "F", "M"]:
    share = rfm.groupby(col)["monetary"].sum() / rfm["monetary"].sum()
    print(f"\nShare of net revenue by {col} score:")
    print(share.round(3).to_string())

# 1) Raw distributions (log10 computed manually for skewed variables)
rfm["log10_frequency"] = np.log10(rfm["frequency"])
rfm["log10_monetary"] = np.log10(rfm["monetary"])

fig = px.histogram(rfm, x="recency", nbins=50, title="Distribution of recency (days)")
fig.show()

fig = px.histogram(rfm, x="log10_frequency", nbins=40,
                   title="Distribution of frequency (log10 of invoices)",
                   labels={"log10_frequency": "log10(frequency)  [0 = 1 invoice, 1 = 10, 2 = 100]"})
fig.show()

fig = px.histogram(rfm, x="log10_monetary", nbins=50,
                   title="Distribution of monetary (log10 of GBP)",
                   labels={"log10_monetary": "log10(monetary)  [2 = 100 GBP, 3 = 1,000, 4 = 10,000]"})
fig.show()

# 2) Heatmap: average monetary by R x F
heat = rfm.pivot_table(index="F", columns="R", values="monetary", aggfunc="mean").sort_index(ascending=False)
fig = px.imshow(heat, text_auto=".0f", aspect="auto", color_continuous_scale="Blues",
                title="Average net revenue (GBP) by Recency score x Frequency score",
                labels={"x": "R score (5 = most recent)", "y": "F score", "color": "Avg GBP"})
fig.show()

# 3) Revenue share by RFM_Sum
by_sum = rfm.groupby("RFM_Sum").agg(customers=("CustomerID", "size"), revenue=("monetary", "sum")).reset_index()
by_sum["revenue_share"] = by_sum["revenue"] / by_sum["revenue"].sum()
fig = px.bar(by_sum, x="RFM_Sum", y="revenue_share",
             title="Share of net revenue by RFM sum (3 = worst, 15 = best)",
             labels={"revenue_share": "Revenue share"})
fig.update_yaxes(tickformat=".0%")
fig.show()

# 4) Top 10 RFM strings by revenue share
top = rfm.groupby("RFM_Score").agg(customers=("CustomerID", "size"), revenue=("monetary", "sum"),
                                   median_recency=("recency", "median")).reset_index()
top["revenue_share"] = top["revenue"] / top["revenue"].sum()
print("\nTop 10 RFM strings by revenue share:")
print(top.nlargest(10, "revenue_share").round(3).to_string(index=False))


Share of net revenue by R score:
R
1    0.054
2    0.092
3    0.129
4    0.200
5    0.525

Share of net revenue by F score:
F
1    0.040
2    0.047
3    0.088
4    0.145
5    0.681

Share of net revenue by M score:
M
1    0.016
2    0.036
3    0.069
4    0.141
5    0.737



Top 10 RFM strings by revenue share:
RFM_Score  customers    revenue  median_recency  revenue_share
      555        346 3776049.25             5.0          0.457
      455        174 1007054.53            22.5          0.122
      355         81  395709.54            45.0          0.048
      444        114  154036.78            22.0          0.019
      345         42  133930.72            49.5          0.016
      445         39  132182.26            25.0          0.016
      554         83  125641.35             6.0          0.015
      545         39  123727.56             8.0          0.015
      344         87  119768.29            47.0          0.014
      544         85  117348.71             5.0          0.014


### Insights 10 (RFM interpretation)

44. **Revenue is highly concentrated in the top scores.** Net revenue share for score 5 vs score 1: Recency 52.5% vs 5.4%, Frequency 68.1% vs 4.0%, Monetary 73.7% vs 1.6%.
45. **Recent, frequent customers are the most valuable.** In the R x F heatmap, customers with F=5 and R=5 average ~8,941 GBP, versus ~2,331 GBP for F=5 and R=1. At low Frequency (F=1-2) average revenue is small (~280-620 GBP) and changes little with Recency.
46. **The top profile carries almost half of revenue.** `555` (346 customers, 8.0%) generates 45.7% of net revenue; `555`, `455`, and `355` together (601 customers, ~14%) generate about 63%.
47. **A high-value group is drifting away.** `455` and `355` have the same top Frequency and Monetary scores as `555` but median recency of 22.5 and 45 days (vs 5), making them candidates for retention actions.
48. **Revenue share rises steeply with RFM sum.** `RFM_Sum` = 15 (only the `555` profile) accounts for 45.7% of net revenue.
49. **Recency has a large active core and a long flat tail.** About 600 customers bought within the last ~10 days, while customers inactive for 100-374 days appear at a roughly constant rate (read from the chart).

Findings are descriptive; they do not show that recency or frequency cause higher spending.

## Save RFM Table

The customer-level features and RFM scores are saved for `03_clustering.ipynb` and the Streamlit app.

In [7]:
customer_table = feat.merge(
    rfm[["CustomerID", "R", "F", "M", "RFM_Score", "RFM_Sum"]],
    on="CustomerID",
    how="inner",
)
customer_table["CustomerID"] = customer_table["CustomerID"].astype(str)

customer_table.to_parquet("../data/customer_rfm.parquet", index=False)

check = pd.read_parquet("../data/customer_rfm.parquet")
print("Saved customers:", len(check))
print("Matches rfm:", len(check) == len(rfm))
print("Columns:", list(check.columns))

Saved customers: 4320
Matches rfm: True
Columns: ['CustomerID', 'frequency', 'gross_revenue', 'avg_order_value', 'avg_units_per_order', 'avg_lines_per_order', 'first_purchase', 'last_purchase', 'recency', 'tenure_days', 'cancelled_value', 'monetary', 'is_uk', 'R', 'F', 'M', 'RFM_Score', 'RFM_Sum']
